In [ ]:
import pandas as pd
import altair as alt

# 1. Cargar el archivo CSV omitiendo las 3 primeras filas de título
df = pd.read_csv("Top5_Balon_de_Oro_2000_2025.csv", skiprows=3)

# 2. Definir los nombres limpios de las columnas
df.columns = ["Ano", "Posicion", "Jugador", "Club", "Nacionalidad", "Champions", "Liga", "Mundial", "Torneo_Continental"]

# 3. Filtrar años válidos (excluyendo 2020 y 2025)
df_valid = df[(df["Ano"] != 2020) & (df["Ano"] != 2025)].copy()

# 4. Recategorizar las posiciones en los tres grupos jerárquicos
def map_pos_group(pos):
    if pos == 1:
        return "1º Balón de Oro"
    elif pos in [2, 3]:
        return "2º y 3º (Podio)"
    else:
        return "4º y 5º lugar"

df_valid["Grupo_Posicion"] = df_valid["Posicion"].apply(map_pos_group)
df_valid["Gano_Champions"] = df_valid["Champions"].astype(str).str.startswith("Sí")

# 5. Calcular los porcentajes por cada grupo de posición
summary = df_valid.groupby(["Grupo_Posicion", "Gano_Champions"]).size().reset_index(name="Cantidad")
total_by_group = df_valid.groupby("Grupo_Posicion").size().reset_index(name="Total")
summary = pd.merge(summary, total_by_group, on="Grupo_Posicion")
summary["Porcentaje"] = (summary["Cantidad"] / summary["Total"] * 100).round(1)
summary["UCL_Label"] = summary["Gano_Champions"].map({
    True: "Ganó Champions",
    False: "No ganó Champions"
})

# 6. Crear el gráfico de barras apiladas con porcentajes en Altair
alt.Chart(summary).mark_bar().encode(
    x=alt.X('Porcentaje:Q', title='Porcentaje de jugadores (%)', scale=alt.Scale(domain=[0, 100])),
    y=alt.Y('Grupo_Posicion:N', title='Posición en el Top 5', sort=['1º Balón de Oro', '2º y 3º (Podio)', '4º y 5º lugar']),
    color=alt.Color('UCL_Label:N', title='Champions League', scale=alt.Scale(range=['#1F4E79', '#B0C4DE'])),
    tooltip=['Grupo_Posicion:N', 'UCL_Label:N', 'Cantidad:Q', 'Porcentaje:Q']
).properties(
    title='Impacto de la Champions League en el Balón de Oro (2000-2025)',
    width=600,
    height=250
)

alt.Chart(...)